# Typing Practice — Chapter 03: Machine Learning Fundamental

Proyek ini melatih memori ototmu: membangun solusi ML dari baris pertama, tanpa mencontek dari notebook utama. Setiap soal adalah satu cell markdown dengan instruksi singkat, lalu cell kode kosong yang harus kamu isi sendiri.

Cara pakainya: baca instruksi di cell markdown, lalu di cell kode di bawahnya, **ketik kode solusi kamu dari nol** — bahkan kalau kamu sudah hafal sintaksnya. Proses mengetik (bukan copy-paste) adalah yang melatih koneksi jari-ke-otak.

Setelah kamu selesai dan jalan dengan benar, cell markdown berikutnya akan muncul berisi **referensi singkat** yang bisa kamu bandingkan dengan kodemu. Tidak harus sama persis — yang penting logikanya benar dan bisa dijalankan. Ada 10 soal bertingkat, dari train/test split manual sampai end-to-end mini pipeline.

---

## Soal 1: Train/Test Split Manual

Tujuan: benar-benar paham apa yang terjadi saat `train_test_split` dipanggil. Tanpa pakai sklearn, buatlah array `X` shape (10, 2) berisi angka 1-20 dan `y` shape (10,) berisi 0/1 selang-seling. Acak index dengan `np.random.permutation`, lalu bagi jadi 80% train dan 20% test. Cetak shape X_train, X_test dan 5 label y_train pertama.

**Referensi singkat untuk Soal 1**:

```python
import numpy as np
np.random.seed(42)
X = np.arange(1, 21).reshape(10, 2)
y = np.array([0, 1] * 5)
idx = np.random.permutation(10)
split = int(0.8 * 10)
train_idx, test_idx = idx[:split], idx[split:]
X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
print(f"y_train[:5]: {y_train[:5]}")
```

`np.random.permutation(n)` mengembalikan array index 0 sampai n-1 yang sudah diacak. `int(0.8 * n)` adalah titik split. Indexing `X[idx]` adalah advanced indexing NumPy — bisa pakai array index untuk mengambil subset.

---

## Soal 2: StandardScaler Manual

Tujuan: paham rumus scaling. Tanpa pakai sklearn, hitung mean dan std `X_train` (axis=0), lalu buat `X_train_scaled = (X_train - mean) / std` dan `X_test_scaled = (X_test - mean_train) / std_train` (pakai mean & std dari TRAIN, bukan test). Verifikasi mean X_train_scaled mendekati 0 dan std mendekati 1. Cetak mean dan std X_test_scaled juga.

**Referensi singkat untuk Soal 2**:

```python
mean = X_train.mean(axis=0)
std  = X_train.std(axis=0)
X_train_scaled = (X_train - mean) / std
X_test_scaled  = (X_test  - mean) / std   # pakai mean & std dari train!
print(f"Train mean: {X_train_scaled.mean(axis=0).round(2)}")
print(f"Test  mean: {X_test_scaled.mean(axis=0).round(2)}")
```

Poin krusial: `X_test_scaled` dihitung dengan `mean` dan `std` dari TRAIN, bukan dari test. Itulah inti anti data leakage. Setelah scaling, train mean ≈ 0 dan train std ≈ 1. Test mean tidak harus 0 — dan itu normal.

---

## Soal 3: Pipeline Sederhana

Bangun Pipeline([StandardScaler, LogisticRegression]) pada dataset Iris. Split stratified 80/20, random_state=42. Latih pipeline, cetak akurasi pada test set, dan pastikan akurasi > 0.9. Verifikasi: kalau kamu inspect `pipeline.named_steps`, kamu harusnya melihat 2 langkah: 'standardscaler' dan 'logisticregression'.

**Referensi singkat untuk Soal 3**:

```python
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
pipeline = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000, random_state=42))])
pipeline.fit(X_train, y_train)
print(f"Test acc: {pipeline.score(X_test, y_test):.4f}")
print(f"Steps: {list(pipeline.named_steps.keys())}")
```

`Pipeline.fit` melakukan scaling di internal pada X_train sebelum melatih classifier. `Pipeline.predict` melakukan scaling dengan parameter dari train, lalu memprediksi — tanpa kamu perlu panggil `scaler.transform` secara manual.

---

## Soal 4: Logistic Regression dengan predict_proba

Latih Logistic Regression pada Iris (Pipeline scaler -> clf). Ambil 5 sampel dari X_test, prediksi kelasnya dengan `predict`, dan probabilitas tiap kelas dengan `predict_proba`. Cetak dengan format: untuk setiap sampel, tampilkan "Sampel i: kelas=X, prob=[a, b, c]". Insight apa yang kamu lihat di probabilitas?

**Referensi singkat untuk Soal 4**:

```python
pipeline.fit(X_train, y_train)
y_pred   = pipeline.predict(X_test[:5])
y_proba  = pipeline.predict_proba(X_test[:5])
for i in range(5):
    print(f"Sampel {i}: kelas={y_pred[i]}, prob={y_proba[i].round(2)}")
```

`predict_proba` mengembalikan matriks (n_sampel, n_kelas) — satu kolom probabilitas per kelas. Tiap baris berjumlah ~1.0. Untuk multi-kelas, kelas yang dipilih adalah yang probabilitasnya paling tinggi. Probabilitas yang rendah (misal 0.4/0.3/0.3) menandakan model ragu.

---

## Soal 5: KNN dengan K Berbeda

Buat list of K = [1, 3, 5, 7, 11, 15, 21]. Untuk setiap K, latih Pipeline(scaler, KNeighborsClassifier(n_neighbors=K)) dan catat akurasi test. Cetak sebagai tabel (K, akurasi) dan identifikasi K dengan akurasi tertinggi.

**Referensi singkat untuk Soal 5**:

```python
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier
results = []
for k in [1, 3, 5, 7, 11, 15, 21]:
    p = Pipeline([('scaler', StandardScaler()), ('clf', KNeighborsClassifier(n_neighbors=k))])
    p.fit(X_train, y_train)
    results.append({'K': k, 'test_acc': p.score(X_test, y_test)})
df = pd.DataFrame(results)
print(df)
print(f"K terbaik: {df.loc[df['test_acc'].idxmax(), 'K']}")
```

Pola yang sangat umum di ML: loop over hyperparameter, latih, evaluasi, bandingkan. `idxmax()` mengembalikan index nilai maksimum — padanannya `df.loc[idx]` untuk dapat baris lengkap.

---

## Soal 6: Decision Tree dengan Max Depth

Latih DecisionTreeClassifier dengan max_depth=None, 1, 2, 3, 5, 10. Untuk setiap nilai, cetak train_acc dan test_acc. Buat list/tabel dan identifikasi: di mana mulai terjadi overfitting (train naik, test turun)?

**Referensi singkat untuk Soal 6**:

```python
from sklearn.tree import DecisionTreeClassifier
for d in [None, 1, 2, 3, 5, 10]:
    dt = DecisionTreeClassifier(max_depth=d, random_state=42)
    dt.fit(X_train, y_train)
    print(f"depth={d}: train={dt.score(X_train, y_train):.3f}, test={dt.score(X_test, y_test):.3f}")
```

Untuk Decision Tree, max_depth=None biasanya menyebabkan train_acc=1.00 (hafal) tapi test_acc tidak lebih baik dari max_depth yang terbatas. Itu tanda klasik overfitting. Sweet spot: max_depth antara 3-5 untuk dataset sekecil Iris.

---

## Soal 7: Random Forest dengan Feature Importance

Latih RandomForestClassifier(n_estimators=100, random_state=42) pada Iris. Cetak akurasi test, lalu akses `model.feature_importances_` dan buat DataFrame dengan kolom 'fitur' dan 'importance'. Sort descending, cetak, dan interpretasi: fitur mana yang paling penting?

**Referensi singkat untuk Soal 7**:

```python
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
print(f"Test acc: {rf.score(X_test, y_test):.4f}")
imp = pd.DataFrame({
    'fitur'      : iris.feature_names,
    'importance' : rf.feature_importances_
}).sort_values('importance', ascending=False)
print(imp)
```

Untuk Iris, petal length dan petal width biasanya jauh lebih penting dari sepal measurements. Ini mencerminkan realitas botani: petal adalah bagian yang paling bervariasi antar spesies. Feature importance seperti ini sangat berguna untuk feature selection.

---

## Soal 8: Cross-Validation

Latih Logistic Regression pada seluruh dataset Iris (X, y). Pakai `cross_val_score(model, X, y, cv=5, scoring='accuracy')`. Cetak 5 skor individual, hitung mean dan std. Lalu ganti cv=10, cetak lagi. Apakah mean cv=5 dan cv=10 mirip? Berapa std-nya?

**Referensi singkat untuk Soal 8**:

```python
from sklearn.model_selection import cross_val_score
model = Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000, random_state=42))])
for cv in [5, 10]:
    scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
    print(f"cv={cv}: scores={scores.round(3)}, mean={scores.mean():.3f}, std={scores.std():.3f}")
```

Cross-validation lebih andal dari single train/test split karena model dievaluasi pada 5 (atau 10) split berbeda. Mean mengukur performa umum, std mengukur stabilitas. cv=10 biasanya memberi mean sedikit lebih akurat tapi komputasi 2x lebih banyak.

---

## Soal 9: Confusion Matrix & Classification Report

Latih Random Forest pada Iris. Prediksi pada X_test. Hitung confusion matrix dengan `confusion_matrix(y_test, y_pred)` dan cetak classification_report dengan `target_names=iris.target_names`. Visualisasikan confusion matrix dengan `ConfusionMatrixDisplay.from_estimator`.

**Referensi singkat untuk Soal 9**:

```python
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=iris.target_names))
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_estimator(rf, X_test, y_test, display_labels=iris.target_names, ax=ax)
plt.show()
```

Confusion matrix diagonal = prediksi benar, off-diagonal = prediksi salah. Baris adalah kelas aktual, kolom adalah kelas prediksi (atau sebaliknya — selalu cek dokumentasi). Classification report memecah precision/recall/F1 per kelas, sangat berguna untuk dataset tidak seimbang.

---

## Soal 10: Mini End-to-End ML Pipeline

Tantangan akhir. Bangun pipeline lengkap dari nol: (1) load Breast Cancer dataset, (2) split stratified 80/20, (3) definisikan 3 model (LogReg, KNN, RF) sebagai Pipeline, (4) cross_val_score(cv=5) untuk masing-masing, (5) pilih model terbaik, (6) fit di train, prediksi di test, (7) cetak classification_report. Target: 30-40 baris, semua dalam satu cell.

**Referensi singkat untuk Soal 10**:

```python
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import numpy as np
import pandas as pd

X, y = load_breast_cancer(return_X_y=True)
target_names = ['ganas', 'jinak']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

models = {
    'LogReg' : Pipeline([('s', StandardScaler()), ('c', LogisticRegression(max_iter=1000, random_state=42))]),
    'KNN'    : Pipeline([('s', StandardScaler()), ('c', KNeighborsClassifier(n_neighbors=5))]),
    'RF'     : Pipeline([('c', RandomForestClassifier(n_estimators=100, random_state=42))]),
}

results = {}
for name, m in models.items():
    scores = cross_val_score(m, X_train, y_train, cv=5, scoring='accuracy')
    results[name] = scores.mean()
print(pd.Series(results).sort_values(ascending=False).round(4))

best_name = max(results, key=results.get)
best_model = models[best_name]
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)
print(f"\nModel terbaik: {best_name}")
print(f"Test acc: {best_model.score(X_test, y_test):.4f}")
print(classification_report(y_test, y_pred, target_names=target_names))
```

Pola ini adalah 90% dari pekerjaan data scientist harian. Kalau kamu bisa menulisnya dari memori dalam 5-10 menit, kamu sudah punya fondasi yang kuat untuk chapter 04 (Hyperparameter Tuning, Imbalanced Data, Time Series, Ensemble Lanjutan).